# Basic Data Analysis of the PHM Dataset

In [ ]:
# Imports
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
from tqdm.notebook import tqdm
import seaborn as sns


#### Load csv with pandas and display it

In [ ]:
milling_df_small = pd.read_parquet('/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/data/phm2007_milling_normalized_small.parquet')
milling_df_small.head()

## Visualization

#### Do some simple Plotting of the data

#### Interactive Plot with Wear Filter
Here is a more advanced interactive plot. You can select a flute, a range for the wear of that flute, and decide whether to include cuts where wear is not available (`NaN`). The available `cut_ID` values will be updated based on your selection.

In [ ]:
# Define the function to plot signals for a given case
def plot_signals_for_case(case_id):
    case_data = milling_df_small[milling_df_small['CaseID'] == case_id]
    
    if case_data.empty:
        print(f"No data found for Case ID: {case_id}")
        return

    # Get signal columns (those with array data)
    signal_cols = [col for col in case_data.columns if isinstance(case_data[col].iloc[0], np.ndarray)]
    
    if not signal_cols:
        print("No signal data with arrays found for this case.")
        return
        
    for signal in signal_cols:
        plt.figure(figsize=(15, 4))
        ax = plt.gca()
        
        # Plot each cut's signal array for the current signal type
        for idx, row in case_data.iterrows():
            # Check if data is a numpy array and not empty
            if isinstance(row[signal], np.ndarray) and row[signal].size > 0:
                ax.plot(row[signal], label=f"Cut {row['run']}")
        
        ax.set_title(f'{signal} for Case {case_id}')
        ax.set_ylabel(signal)
        ax.set_xlabel('Time Step')
        ax.legend(title='Cut', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

# Get unique case IDs
case_ids = sorted(nasa_df['CaseID'].unique())

# Create an interactive dropdown widget
interact(plot_signals_for_case, case_id=widgets.Dropdown(options=case_ids, description='Case ID:'));

### Remove outliers 


In [ ]:
# No outliers detected yet

### More Statistics

Let's first start with some boxplots. of the signals. We keep the "grouping" widgets from the settings df from above.

In [ ]:
def signal_boxplot(grouping, sample_frac=1.0):
    """
    Generates a boxplot of signal distributions, with optional sampling for efficiency.
    
    Parameters:
    - grouping (str): The column to group by (e.g., 'cutter').
    - sample_frac (float): The fraction of data to sample from each signal array. 
                             1.0 means no sampling.
    """
    # The signals to be plotted
    signal_list = ['V_mag', 'AE_RMS']
    
    # Create a new DataFrame in long format by "exploding" the arrays
    long_format_dfs = []
    for signal in signal_list:
        # Create a temporary DataFrame for the current signal
        temp_df = milling_df_small[[grouping, signal]].copy()
        
        # If sampling, sample from each array in the signal column
        if sample_frac < 1.0:
            sample_size = lambda x: max(1, int(len(x) * sample_frac))
            temp_df[signal] = temp_df[signal].apply(lambda x: np.random.choice(x, size=sample_size(x), replace=False))

        # Explode the array column into multiple rows
        temp_df = temp_df.explode(signal)
        temp_df['Signal'] = signal  # Add a column to identify the signal
        temp_df.rename(columns={signal: 'Value'}, inplace=True)
        long_format_dfs.append(temp_df)
    
    # Concatenate all the long-format dataframes
    melted_df = pd.concat(long_format_dfs, ignore_index=True)
    
    # Ensure the 'Value' column is numeric for plotting
    melted_df['Value'] = pd.to_numeric(melted_df['Value'])

    # Create the boxplot with the long-form data
    plt.figure(figsize=(12, 8)) # Add a figure size for better readability
    sns.boxplot(data=melted_df, x='Signal', y='Value', hue=grouping)
    title = f'Signal Distribution by {grouping}'
    if sample_frac < 1.0:
        title += f' (Sampled {sample_frac*100:.0f}%)'
    plt.title(title)
    plt.show()

In [ ]:
# To speed up the process, you can use a sample of the data.
# For example, using 10% of the data:
signal_boxplot('cutter', sample_frac=0.1)